**How to Run**
- First you should change the runtime from CPU to T4 GPU
- then run the step 1 it will ask for the permission to the drive provide it full authority
- step 3 is the process of importing the hugging face model
- step 4 is running on the images of cats and dogs i collected there are 2 oversize width images and model should be able to detect it
- Step 5 takes atleast 10 min to execute
- in step 7 you should replace you real prompt in there
- for the cell after step 5 run them as they are written so step 7 then step 8 then step 6 and atlast step 9
- if you run it out of order it will show you errors and sometimes you have to run it again from the start because the connection from drive disconnects after a while
- steps 5 and other have high GPU usage and it will displayed in the printed messages but you can also see it on the top left RAM and disk usage


In [20]:
#step 2 - run this after step 1 so you can connect your drive with google collab
#make sure that there are folder vision_outputs and test_images in the Lighthouse folder

from google.colab import drive
drive.mount('/content/drive')

import os
BASE = "/content/drive/MyDrive/Lighthouse"
os.makedirs(f"{BASE}/test_images", exist_ok=True)      # images the model will look at
os.makedirs(f"{BASE}/vision_outputs", exist_ok=True)   # results get saved here

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [21]:
#step 3 - this cell should be run after the drive cell (step 2 cell)
#this is cell is responsible for fetching the images from the test_image folder
#comments are there to help you
MODEL_ID   = "Qwen/Qwen2-VL-7B-Instruct"   # leave as is
IMAGE_DIR  = f"{BASE}/test_images"         # Arjun: change to the ContextDP image folder
OUT_DIR    = f"{BASE}/vision_outputs"      # where results are saved; leave as is
OUT_FILE   = "vision_test_run.csv"         # Arjun: change to dev_run.csv, then preds_vision.csv
IMAGES     = []                            # filenames to run, filled in below. Arjun: fill from dev_images.csv, then the test half
PROMPT     = ('What is the main thing in this image? '
              'Answer only in JSON: {"main_thing": "...", "has_text": true or false}')
                                           # Arjun: replace with your prompt
MAX_NEW_TOKENS = 200                       # Arjun: raise if replies get cut off

print("Settings loaded. Images from:", IMAGE_DIR, "| Results to:", OUT_DIR)

Settings loaded. Images from: /content/drive/MyDrive/Lighthouse/test_images | Results to: /content/drive/MyDrive/Lighthouse/vision_outputs


In [22]:
#step 4
#this step will be used for making a labels.csv in which there
#will be an image, label there are 18 regular images and 2 oversize ones
import os,csv
from PIL import Image
labels_path = f"{IMAGE_DIR}/labels.csv"
#PIL will be used for opening and reading the images width and height
#IMAGES_DIR is from the setting where the images are

def label_from_name(fname):
  stem = os.path.splitext(fname)[0].lower()
  #this will the name of the image into two parts one will be the name
  #and the second will be the type in which it was stored like "jpeg"
  return stem.split("_")[0]
  #this is just for the indexing like start at the name first not the type it is stored in
  #this will also split the name at the underscore better to cqtegorize first
  #what is the image on

image_files = sorted(f for f in os.listdir(IMAGE_DIR)
                      if f.lower().endswith((".jpg",".jpeg")))
#this will be used for returning every name of the file in the folder as a list
#the order of this will be random
#the for loop will then run through it and end if the storage type is one of those extensions
#case-sensitivity is omitted because of the .lower part

with open(labels_path, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["image", "true_label"])
    for name in image_files:
        w.writerow([name, label_from_name(name)])
#in this we will be opening files for writing every run we will create a new csv file
#the csv writer will be used for turning the python list to csv files


with open(labels_path) as f:
  IMAGES = [row["image"] for row in csv.DictReader(f)]

big = 0
for name in IMAGES:
  width, height = Image.open(f"{IMAGE_DIR}/{name}").size
  big+= width>2000
  print(f"{name:25s} {label_from_name(name):8s} {width}*{height}")
#this part is for checking the large images
#if the pixel width is greater than 2000 there will be a boolean answer


print(f"\n{len(IMAGES)} images, {big} files are over 2000 pixels of width")
assert big>=2, "there should be atleast two images over 2000 pixels of width"


cat_01.jpg                cat      500*553
cat_02.jpg                cat      736*633
cat_03.jpg                cat      1080*1080
cat_04.jpg                cat      530*640
cat_05.jpg                cat      210*210
cat_06.jpg                cat      1795*2397
cat_07.jpg                cat      1920*1280
cat_08.jpg                cat      3872*2592
cat_09.jpeg               cat      225*225
cat_10.jpeg               cat      275*183
dog_01.jpeg               dog      275*183
dog_02.jpeg               dog      275*183
dog_03.jpeg               dog      275*183
dog_04.jpeg               dog      275*183
dog_05.jpeg               dog      300*168
dog_06.jpeg               dog      238*212
dog_07.jpg                dog      3176*2117
dog_08.jpeg               dog      163*308
dog_09.jpeg               dog      183*275
dog_10.jpeg               dog      168*300

20 images, 2 files are over 2000 pixels of width


In [23]:
#Step 5
!pip install -q -U transformers accelerate bitsandbytes

#transformer is there for running the hugging face model here
#the bitsandbytes is just for 4 bit compression just fulfilling the requirement

import transformers, bitsandbytes
print("transformers",transformers.__version__," , bitsandbytes", bitsandbytes.__version__)

import torch
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor , BitsAndBytesConfig

bites_config = BitsAndBytesConfig(
    load_in_4bit= True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bites_config,
    device_map= "auto",
    torch_dtype=torch.float16,
)
model.eval()

processor= AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pix=256*28*28,
    max_pix=1024*28*28,
)

print("model is loaded on", model.device)
print(f"GPU updated memory after the load is: {torch.cuda.memory_allocated()/1024**3:.2f} GB")


transformers 5.18.0  , bitsandbytes 0.50.2


Loading weights:   0%|          | 0/730 [00:00<?, ?it/s]

model is loaded on cuda:0
GPU updated memory after the load is: 9.03 GB


In [24]:
# Step 7 - Read the replies
#run this before step 6
# Pulls the JSON out of each model reply. If it can't, returns "parse_fail".
import json, re

# Arjun: replace this with your parser for the real prompt
def parse_reply(reply):
    try:
        match = re.search(r"\{.*\}", reply, re.DOTALL)   # find the {...} part
        data = json.loads(match.group(0))                  # turn it into a Python dict
        return data["main_thing"], data["has_text"]
    except Exception:
        return "parse_fail", "parse_fail"

# Quick self-test so this cell shows output
print(parse_reply('{"main": "cat", "text": false}'))
print(parse_reply('```json\n{"main_thing": "dog", "has_text": true}\n```'))
print(parse_reply('I think this is a cat'))

('parse_fail', 'parse_fail')
('dog', True)
('parse_fail', 'parse_fail')


In [25]:
# Step 8 - Save the results
# Defines save_results(), which Step 6 calls every 5 images (save as you go).
# Parses each raw reply with parse_reply() and writes OUT_DIR/OUT_FILE.
import os
import pandas as pd

out_path = os.path.join(OUT_DIR, OUT_FILE)
COLUMNS = ["image", "raw_reply", "main_thing", "has_text", "seconds"]
SAVE_EVERY = 5

def save_results(results):
    rows = []
    for r in results:
        main_thing, has_text = parse_reply(r["raw_reply"])
        rows.append({"image": r["image"], "raw_reply": r["raw_reply"],
                     "main_thing": main_thing, "has_text": has_text,
                     "seconds": r["seconds"]})
    df = pd.DataFrame(rows, columns=COLUMNS)
    df.to_csv(out_path, index=False)
    return df

print(f"Results will be saved to {out_path} every {SAVE_EVERY} images")

Results will be saved to /content/drive/MyDrive/Lighthouse/vision_outputs/vision_test_run.csv every 5 images


In [26]:
# Step 6 - Run the images
# Sends each image + PROMPT to the model, keeps the raw reply and seconds,
# saves every SAVE_EVERY images (Step 8), then records peak GPU memory.
import time

torch.cuda.reset_peak_memory_stats()
results = []

for name in IMAGES:
    t0 = time.time()
    try:
        img = Image.open(os.path.join(IMAGE_DIR, name)).convert("RGB")
        messages = [{"role": "user", "content": [
            {"type": "image"},
            {"type": "text", "text": PROMPT},
        ]}]
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = processor(text=[text], images=[img], return_tensors="pt").to(model.device)

        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
        torch.cuda.synchronize()

        new_tokens = out[:, inputs["input_ids"].shape[1]:]
        reply = processor.batch_decode(new_tokens, skip_special_tokens=True)[0].strip()
    except Exception as e:
        reply = f"ERROR: {e!r}"             # keep the row; parse_reply will mark it parse_fail

    secs = round(time.time() - t0, 2)
    results.append({"image": name, "raw_reply": reply, "seconds": secs})
    print(f"{name}: {secs}s -> {reply[:80]}")

    if len(results) % SAVE_EVERY == 0:      # save as you go
        save_results(results)
        print(f"   saved {len(results)} rows")

df = save_results(results)                  # final save

peak_gb_used = torch.cuda.max_memory_allocated() / 1024**3
print(f"\nSaved {len(df)} rows to {out_path}")
print(f"Peak GPU memory during the run: {peak_gb_used:.2f} GB")
df


cat_01.jpg: 2.76s -> {"main_thing": "A cat", "has_text": false}
cat_02.jpg: 3.92s -> {"main_thing": "A cat", "has_text": true}
cat_03.jpg: 5.93s -> {
  "main_thing": "A cat",
  "has_text": false
}
cat_04.jpg: 2.73s -> {"main_thing": "A cat", "has_text": false}
cat_05.jpg: 3.68s -> {"main_thing": "A cat", "has_text": false}
   saved 5 rows
cat_06.jpg: 14.41s -> ERROR: OutOfMemoryError('CUDA out of memory. Tried to allocate 3.22 GiB. GPU 0 h
cat_07.jpg: 13.89s -> {"main_thing": "A cat", "has_text": false}
cat_08.jpg: 96.59s -> ERROR: OutOfMemoryError('CUDA out of memory. Tried to allocate 17.32 GiB. GPU 0 
cat_09.jpeg: 3.56s -> {
  "main_thing": "A cat",
  "has_text": false
}
cat_10.jpeg: 5.49s -> {"main_thing": "A cat", "has_text": false}
   saved 10 rows
dog_01.jpeg: 4.38s -> {"main_thing": "dog", "has_text": false}
dog_02.jpeg: 5.07s -> {
  "main_thing": "dog",
  "has_text": false
}
dog_03.jpeg: 3.62s -> {
  "main_thing": "dog",
  "has_text": false
}
dog_04.jpeg: 1.17s -> {"main_thing

,image,raw_reply,main_thing,has_text,seconds
0,cat_01.jpg,"{""main_thing"": ""A cat"", ""has_text"": false}",A cat,False,2.76
1,cat_02.jpg,"{""main_thing"": ""A cat"", ""has_text"": true}",A cat,True,3.92
2,cat_03.jpg,"{\n ""main_thing"": ""A cat"",\n ""has_text"": fal...",A cat,False,5.93
3,cat_04.jpg,"{""main_thing"": ""A cat"", ""has_text"": false}",A cat,False,2.73
4,cat_05.jpg,"{""main_thing"": ""A cat"", ""has_text"": false}",A cat,False,3.68
5,cat_06.jpg,ERROR: OutOfMemoryError('CUDA out of memory. T...,parse_fail,parse_fail,14.41
6,cat_07.jpg,"{""main_thing"": ""A cat"", ""has_text"": false}",A cat,False,13.89
7,cat_08.jpg,ERROR: OutOfMemoryError('CUDA out of memory. T...,parse_fail,parse_fail,96.59
8,cat_09.jpeg,"{\n ""main_thing"": ""A cat"",\n ""has_text"": fal...",A cat,False,3.56
9,cat_10.jpeg,"{""main_thing"": ""A cat"", ""has_text"": false}",A cat,False,5.49


In [27]:
# Step 9 - Check the results against the true labels
import pandas as pd

res = pd.read_csv(os.path.join(OUT_DIR, OUT_FILE))
labels = pd.read_csv(os.path.join(IMAGE_DIR, "labels.csv"))

# 1. Match results to labels by image name
merged = labels.merge(res, on="image", how="left")

# 2. Print true label next to what the model said
for _, row in merged.iterrows():
    print(f"{row['image']:20s} true: {row['true_label']:6s} model: {row['main_thing']}")

# 3. Count answers that mention the true label
merged["match"] = [str(t).lower() in str(m).lower()
                   for t, m in zip(merged["true_label"], merged["main_thing"])]
n_match = merged["match"].sum()

# 4. Count parse failures
n_fail = (merged["main_thing"] == "parse_fail").sum()

print(f"\nMatches: {n_match} / {len(merged)}")
print(f"Parse fails: {n_fail}")

print(f"\nAvg seconds per image: {res['seconds'].mean():.1f}")
print(f"Peak GPU memory: {peak_gb_used:.2f} GB")

cat_01.jpg           true: cat    model: A cat
cat_02.jpg           true: cat    model: A cat
cat_03.jpg           true: cat    model: A cat
cat_04.jpg           true: cat    model: A cat
cat_05.jpg           true: cat    model: A cat
cat_06.jpg           true: cat    model: parse_fail
cat_07.jpg           true: cat    model: A cat
cat_08.jpg           true: cat    model: parse_fail
cat_09.jpeg          true: cat    model: A cat
cat_10.jpeg          true: cat    model: A cat
dog_01.jpeg          true: dog    model: dog
dog_02.jpeg          true: dog    model: dog
dog_03.jpeg          true: dog    model: dog
dog_04.jpeg          true: dog    model: dog
dog_05.jpeg          true: dog    model: A dog
dog_06.jpeg          true: dog    model: dog
dog_07.jpg           true: dog    model: parse_fail
dog_08.jpeg          true: dog    model: A black and white dog
dog_09.jpeg          true: dog    model: dog
dog_10.jpeg          true: dog    model: A black and white dog

Matches: 17 / 20
Parse f